In [ ]:
!pip install evidently
!pip install "modin[all]" # (Recommended) Install Modin with Ray and Dask engines.

import modin.config as modin_cfg

modin_cfg.Engine.put("ray")  # Modin will use Ray
import modin.pandas as pd

In [ ]:
import kagglehub
path = kagglehub.dataset_download("shivamb/netflix-shows")

In [ ]:
import os

# Assuming the CSV file is named 'netflix_titles.csv' inside the downloaded directory.
# You might need to adjust 'netflix_titles.csv' if the actual file name is different.
csv_file_path = os.path.join(path, 'netflix_titles.csv')
df = pd.read_csv(csv_file_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Fill NaN values in relevant columns to avoid errors during text processing
df['listed_in'] = df['listed_in'].fillna('')
df['description'] = df['description'].fillna('')
df['director'] = df['director'].fillna('')
df['cast'] = df['cast'].fillna('')

# Combine relevant text features into a single string for each item
# This creates a 'soup' of keywords for each movie/show
df['features'] = df['listed_in'] + ' ' + df['description'] + ' ' + df['director'] + ' ' + df['cast']

# Initialize TF-IDF Vectorizer
# tfidf = TfidfVectorizer(stop_words='english') # uncomment if you want to remove english stopwords
tfidf = TfidfVectorizer()

# Construct the TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(df['features'])

print(f"Shape of TF-IDF matrix: {tfidf_matrix.shape}")

# Calculate the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Shape of Cosine Similarity matrix: {cosine_sim.shape}")

# Create a Series with movie titles as index for easy lookup
titles = pd.Series(df.index, index=df['title']).drop_duplicates()

# Example function to get recommendations
def get_recommendations(title, cosine_sim=cosine_sim, df=df, titles=titles):
    if title not in titles.index:
        print(f"' {title} ' not found in dataset. Please check the title.")
        return

    # Get the index of the movie that matches the title
    idx = titles[title]

    # Get the pairwise similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar movies (excluding itself)
    sim_scores = sim_scores[1:11]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 10 most similar movies
    return df['title'].iloc[movie_indices]

# Example usage:
print("\nRecommendations for 'Kota Factory':")
print(get_recommendations('Kota Factory'))

print("\nRecommendations for 'Blood & Water':")
print(get_recommendations('Blood & Water'))